# BLIP VQA-base — DIMER visual question answering tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/blip-vqa-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/blip-vqa-pipeline/blob/main/tutorials/blip_vqa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Salesforce%2Fblip--vqa--base-ffcc4d?style=flat)](https://huggingface.co/Salesforce/blip-vqa-base) [![Upstream](https://img.shields.io/badge/Upstream-salesforce%2FBLIP-181717?style=flat&logo=github&logoColor=white)](https://github.com/salesforce/BLIP) [![arXiv](https://img.shields.io/badge/arXiv-2201.12086-b31b1b.svg)](https://arxiv.org/abs/2201.12086)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** Visual question answering — one image plus one natural-language question → one short answer string — using the pinned `Salesforce/blip-vqa-base` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/blip_vqa_pipeline/pipeline.py` at revision `40bf9c94e930`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `787b3d35d57e49572baabd22884b3d5a05acf072` (~1540 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the BLIP model (a ViT-B/16 image encoder at 384×384, a BERT-style question encoder that attends to the image features, and a 12-layer answer decoder; about 385M parameters, pretrained on 129M image–text pairs with captioning-and-filtering bootstrapping and fine-tuned on VQA v2) encodes the resized image, encodes the question against it, and generates the answer text token by token. Decoding is greedy (`do_sample=False`) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, a non-empty question up to 256 characters, the token budget), a fixed output contract, and the `exact_match`, `anls`, `vqa_accuracy`, `validate_inputs` and `evaluation_report` helpers. The default sample is a flat cartoon scene drawn in code with seven authored questions and accepted answers, so exact-match and ANLS are demonstration (plumbing) evidence for one drawing, not a VQA benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic scene with authored question/answer pairs (or upload your own photograph and write your own questions) and validate it into an input manifest, choose a token budget, run the supported task, read the answers correctly (generated text, no score, a `truncated` flag), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `exact_match` and `anls` only when accepted answers exist and `not-measurable` otherwise, and export the answers, the annotated image and provenance.

**This notebook does not demonstrate:** Reading text in the image (BLIP-VQA is not an OCR or document model; use a document-QA pipeline for that), counting beyond a few objects or spatial reasoning the model was not trained for, answer localisation (the model returns text, not a region), open-ended captioning (a separate checkpoint), batch throughput, sampling or beam search, evaluation on the VQA v2 benchmark (not bundled; only authored questions on a drawn scene are scored here), and any training. The model was fine-tuned on photographs with short English answers; drawings, diagrams, non-English questions and long free-text answers are outside what this notebook measures, and a fluent wrong answer carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.8 s to load and about 0.2 s per question on the 640×480 drawn scene in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 1.54 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; what VQA accuracy (`min(matching human answers / 3, 1)`) and normalised Levenshtein similarity (ANLS) measure; that a confident answer is not a correct one.
- **Data:** the default sample is a deterministic 640×480 cartoon scene drawn in code with Pillow (sky, grass, sun, a red house with a brown door, one tree, a white ball; no text rendering, so its digest is stable across Pillow builds) with seven authored questions and their accepted answers, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px, plus your own questions typed into the form field. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Salesforce/blip-vqa-base` snapshot (~1540 MB in total) at revision `787b3d35d57e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'blip-vqa-pipeline',
    'repository_revision': '40bf9c94e93013d40c7cc9da4c02606addfc9c98',
    'embedded_module': 'src/blip_vqa_pipeline/pipeline.py',
    'embedded_modules': ['src/blip_vqa_pipeline/pipeline.py'],
    'module_sha256': '1c5649413e97fab3194b3ec0985fbfd5604002e3a69661179d78346c79c1c928',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/blip_vqa_pipeline/` @ `40bf9c94e930`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/blip_vqa_pipeline/pipeline.py`

In [ ]:
"""Visual question answering with the pinned ``Salesforce/blip-vqa-base`` checkpoint (BLIP, ViT-B).

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the BLIP architecture comes from the pinned ``transformers`` release, the
weights are SafeTensors, and no model-repository code is executed.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "Salesforce/blip-vqa-base"
MODEL_REVISION = "787b3d35d57e49572baabd22884b3d5a05acf072"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "blip-vqa-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Generation ceilings. VQA answers are one to a few words (the upstream README example decodes
# `model.generate(**inputs)` at its default length); the default leaves room for a short phrase and
# the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 32
DEFAULT_MAX_NEW_TOKENS = 10
DECODING = "greedy"
# Question ceiling. The BERT tokenizer truncates the question at the text encoder's positions; a
# question far longer than a sentence is not what the model was trained on.
MAX_QUESTION_CHARS = 256
# Input ceilings. The processor resizes every image to 384x384 (preprocessor_config.json, aspect
# ratio not preserved) into 24x24 = 576 ViT-B/16 patches, so image cost is bounded; the side ceiling
# only guards memory during decoding and resizing.
IMAGE_SIZE = 384
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# VQA accuracy (the VQA v2 convention): min(#matching accepted answers / 3, 1).
VQA_ACCURACY_DIVISOR = 3
# ANLS (a relaxed string similarity from DocVQA, reported alongside): below this threshold scores 0.
ANLS_THRESHOLD = 0.5
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_answer(text: str) -> str:
    """DocVQA-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def _levenshtein(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def anls(prediction: str, golds: Sequence[str], *, threshold: float = ANLS_THRESHOLD) -> float:
    """Average Normalised Levenshtein Similarity for one question (Biten et al., ICDAR 2019).

    ``1 - lev(pred, gold) / max(len(pred), len(gold))`` over normalised strings, maximised over the
    accepted ``golds``; a similarity below ``threshold`` scores 0 so a near-miss is not rewarded.
    """
    if not golds:
        raise ValueError("golds must contain at least one accepted answer")
    pred = normalize_answer(prediction)
    best = 0.0
    for gold in golds:
        ref = normalize_answer(gold)
        longest = max(len(pred), len(ref))
        similarity = 1.0 if longest == 0 else 1.0 - _levenshtein(pred, ref) / longest
        best = max(best, similarity)
    return best if best >= threshold else 0.0


def exact_match(prediction: str, golds: Sequence[str]) -> bool:
    """Whether the normalised prediction equals any normalised accepted answer."""
    pred = normalize_answer(prediction)
    return any(pred == normalize_answer(gold) for gold in golds)


def vqa_accuracy(prediction: str, golds: Sequence[str]) -> float:
    """VQA v2 accuracy for one question: min(number of accepted answers the prediction matches / 3, 1).

    With the benchmark's ten human answers a prediction three humans gave scores 1.0; with one to three
    authored answers the measure degenerates to a graded exact match and is stated as such.
    """
    if not golds:
        raise ValueError("golds must contain at least one accepted answer")
    pred = normalize_answer(prediction)
    matches = sum(pred == normalize_answer(gold) for gold in golds)
    return min(matches / VQA_ACCURACY_DIVISOR, 1.0)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB) plus one question string",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "image resized to 384x384 (aspect ratio not preserved, CLIP mean/std) into 576 ViT-B/16 patches; "
        "the question is tokenised by the snapshot's BERT tokenizer and encoded against the image "
        "features; the answer decoder generates the answer text"
    ),
    "output": "one answer string (the model's decoded text), no score",
}


def _check_inputs(image: Any, question: Any, max_new_tokens: Any) -> tuple[Image.Image, str, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``answer`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked_question = " ".join(question.split())
    if not checked_question:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked_question) > MAX_QUESTION_CHARS:
        raise ValueError(
            f"question has {len(checked_question)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}"
        )
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_question, max_new_tokens


def validate_inputs(
    image: Image.Image,
    questions: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every question is checked exactly as ``answer`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(questions, str) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of str")
    checked = [_check_inputs(image, question, max_new_tokens)[1] for question in questions]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "questions": checked,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    golds: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one sequence of accepted answers per result, in order) the report carries the
    mean ``anls`` and the ``exact_match`` rate over the questions plus one per-question entry, verdict
    ``sample-sanity``; without golds it is ``not-measurable`` and says what labelled data would make
    the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "image + question -> short answer text (visual question answering)",
        "score_semantics": (
            "the answer is generated text and carries no score, probability or correctness signal; a "
            "fluent answer is not evidence that it describes the image. Greedy decoding makes the output "
            "reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no accepted answers were supplied for the evaluated questions",
            "needs": (
                "question/answer pairs with accepted answers on images from the deployment domain "
                "(VQA v2-style annotations, ten answers per question) scored with VQA accuracy; no such "
                "labelled set ships with this repository"
            ),
        }
    if len(golds) != len(results):
        raise ValueError(f"golds has {len(golds)} entries for {len(results)} results")
    per_question = []
    for result, accepted in zip(results, golds, strict=True):
        if isinstance(accepted, str) or not accepted:
            raise ValueError("each golds entry must be a non-empty sequence of accepted answers")
        prediction = str(result["answer"])
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": prediction,
                "golds": list(accepted),
                "exact_match": exact_match(prediction, accepted),
                "anls": anls(prediction, accepted),
            }
        )
    metrics = [
        {
            "id": "exact_match",
            "value": sum(entry["exact_match"] for entry in per_question) / len(per_question),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; any accepted answer",
            "relation_to_vqa_accuracy": (
                "VQA accuracy (min(matching human answers / 3, 1)) needs several human answers per "
                "question; with authored accepted answers it degenerates to this exact match"
            ),
            "estimation": f"{len(per_question)} question(s) on one image, no dispersion estimate",
        },
        {
            "id": "anls",
            "value": sum(entry["anls"] for entry in per_question) / len(per_question),
            "threshold": ANLS_THRESHOLD,
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; max over golds",
            "estimation": f"{len(per_question)} question(s) on one image, no dispersion estimate",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial image you drew yourself; plumbing "
            "evidence, not a VQA benchmark"
        ),
        "needs": (
            "a labelled question/answer set on images from the deployment domain (cameras, scenes, "
            "question styles) with several accepted answers per question for any accuracy claim; VQA v2 "
            "is not bundled"
        ),
    }


@dataclass
class BlipVQAPipeline:
    """``_runner(image, question, max_new_tokens)`` returns ``{"answer": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BlipVQAPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import BlipForQuestionAnswering, BlipProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = BlipProcessor.from_pretrained(location, **common)
        model = BlipForQuestionAnswering.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, question: str, max_new_tokens: int) -> dict[str, Any]:
            inputs = processor(images=image, text=question, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            # The answer decoder's output holds only answer tokens (bos + answer + sep).
            answer_ids = generated[0]
            decoded = processor.decode(answer_ids, skip_special_tokens=True)
            return {"answer": decoded, "new_tokens": max(int(answer_ids.shape[0]) - 1, 0)}

        return cls(runner, resolved_device, "float32", source)

    def answer(
        self,
        image: Image.Image,
        question: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Answer one question about one image; ``answer`` is the decoded text, stripped."""
        rgb, checked_question, checked_tokens = _check_inputs(image, question, max_new_tokens)
        raw = self._runner(rgb, checked_question, checked_tokens)
        if not isinstance(raw, dict) or "answer" not in raw:
            raise RuntimeError("runner must return a dict with 'answer'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "answer": str(raw["answer"]).strip(),
            "question": checked_question,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `787b3d35d57e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BlipVQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "blip-vqa-base",
  "modelId": "Salesforce/blip-vqa-base",
  "revision": "787b3d35d57e49572baabd22884b3d5a05acf072",
  "files": [
    {
      "path": "README.md",
      "bytes": 5459,
      "sha256": "03e051ff0a0461a06892f90ac0e10bd2eebc2dd47c3ffa65092b31576aa37e4c"
    },
    {
      "path": "config.json",
      "bytes": 4559,
      "sha256": "689a09e2a9980b7fcad329271c032254fde23b1ee7a90c67b003e1867dc9c098"
    },
    {
      "path": "model.safetensors",
      "bytes": 1538800584,
      "sha256": "33786eed34def0c95fa948128cb4386be9b9219aa2c2e25f1c9c744692121bb7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 445,
      "sha256": "0aa66e2e9ac3ea3b5cd4388c35072e22db4e1cc1f96c7872bed07749c712ade1"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 125,
      "sha256": "b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 711396,
      "sha256": "d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 592,
      "sha256": "48d1c9120fe61c9741286050189b030e936f23797a65fd0a179078661e1c43bb"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 1539754668
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BlipVQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scene or optional BYOD

The default sample is **synthetic** and carries its own references: a flat cartoon scene — blue sky, green grass, a yellow sun, a red house with a brown roof and a brown door, one round tree and a white ball — is drawn with Pillow at 640×480, the same drawing the repository's smoke run used. Seven questions are authored against it, each with the accepted answer(s) as drawn; they are the references for the `exact_match` and `anls` sanity checks later. Two of them are **recorded misses** from the smoke run, kept on purpose: the model counted two trees where one is drawn and called the brown door `red`. They are not a labelled dataset, so nothing here is a VQA v2 measurement. The image digest is printed for the record. BYOD is optional and disabled by default; when enabled, upload one image and type your questions (one per line) — no accepted answers exist for them, so the evaluation report will be `not-measurable`.

The token budget is a **caller-owned request parameter**: `max_new_tokens` bounds the answer (`DEFAULT_MAX_NEW_TOKENS = 10` fits any VQA-style answer; `MAX_NEW_TOKENS = 32` is the ceiling). Nothing is validated in this cell — the next section hands the image and the questions to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the budget and the number of questions.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
byod_questions = 'what is in the picture?\nwhat color is the largest object?'  # @param {type:"string"}
max_new_tokens = 10  # @param {type:"integer"}


def synthetic_scene(width=640, height=480):
    """A flat cartoon scene drawn with Pillow (no text); returns image + [(question, accepted answers)]."""
    image = Image.new('RGB', (width, height), (135, 206, 235))  # sky
    d = ImageDraw.Draw(image)
    d.rectangle([0, 300, 640, 480], fill=(60, 179, 75))  # grass
    d.ellipse([500, 40, 600, 140], fill=(255, 215, 0))  # sun
    d.rectangle([120, 180, 320, 330], fill=(200, 40, 40))  # red house
    d.polygon([(100, 180), (220, 90), (340, 180)], fill=(90, 50, 20))  # brown roof
    d.rectangle([200, 260, 240, 330], fill=(70, 40, 20))  # brown door
    d.ellipse([420, 260, 520, 360], fill=(40, 100, 40))  # tree crown
    d.rectangle([460, 350, 480, 420], fill=(90, 60, 30))  # trunk
    d.ellipse([60, 380, 140, 440], fill=(255, 255, 255))  # white ball
    qa = [
        ('what color is the house?', ['red']),
        ('how many houses are there?', ['1', 'one']),
        ('what is the yellow object?', ['sun', 'the sun']),
        ('what is next to the house?', ['tree', 'a tree']),
        ('what color is the sky?', ['blue']),
        ('how many trees are there?', ['1', 'one']),  # smoke run answered 2: recorded miss
        ('what color is the door?', ['brown', 'dark brown']),  # smoke run answered red: recorded miss
    ]
    return image, qa


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    questions = [line.strip() for line in byod_questions.splitlines() if line.strip()]
    golds = None
    sample_kind = 'BYOD'
else:
    # Deterministic drawing: no randomness and no text rendering, so no seed is needed and the digest is stable.
    image, qa = synthetic_scene()
    questions, golds = [q for q, _ in qa], [g for _, g in qa]
    image_name = 'synthetic_scene_640x480.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'max_new_tokens': max_new_tokens, 'n_questions': len(questions), 'has_golds': golds is not None})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `answer` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, each question a non-empty string of at most `MAX_QUESTION_CHARS` characters (whitespace collapsed), and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the 384×384 resize that does not preserve aspect ratio, and the decoding rule), the input's observed mode and size, the checked questions, the budget and the verdict. The manifest is written to `outputs/blip_vqa_input_manifest.json`. To show what rejection looks like, the cell also validates a blank question and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB and resized to `IMAGE_SIZE`×`IMAGE_SIZE`; nothing else is dropped or altered. The pipeline cannot tell whether the question is answerable from the image: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'IMAGE_SIZE': IMAGE_SIZE, 'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}})
input_manifest = validate_inputs(image, questions, max_new_tokens=max_new_tokens, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, ['   '])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'blank-question-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/blip_vqa_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Answer the questions and read the output correctly

`answer` returns, per question, a dict with `answer` (the decoded text, stripped), the checked `question`, `image_size`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: the answer is generated text with no probability and no correctness signal, and a fluent answer is not evidence that it describes the image. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection can change a token and therefore the rest of the answer, so GPU and CPU outputs need not match. Each call re-encodes the image with the question, so cost is per question (about 0.2 s each on the reference CPU). As recorded in the model card, the repository's CPU smoke on this same drawing answered five of the seven authored questions exactly, counted `2` trees and called the door `red` — and answered `dog` to "what is in the picture?" on a blank white image: the model always produces an answer, whether or not one exists.

In [ ]:
import time

results, seconds = [], []
for question in questions:
    t0 = time.time()
    results.append(pipe.answer(image, question, max_new_tokens=max_new_tokens))
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'dtype': pipe.dtype, 'seconds_per_question': seconds, 'any_truncated': any(r['truncated'] for r in results)})
for result in results:
    print(f"Q: {result['question']}\n   A: {result['answer']!r}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})")
if any(r['truncated'] for r in results):
    print('A budget was exhausted: that answer is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No accuracy is reported by default: VQA accuracy needs labelled questions with several human answers each on images from the deployment domain, and this repository ships none (VQA v2 is not bundled). The repository's metric helpers are `exact_match` after normalisation (lower-case, punctuation removed, whitespace collapsed) against any accepted answer; `anls` — normalised Levenshtein similarity `1 − edits / max(len)`, maximised over the accepted answers, scored 0 below the 0.5 threshold — reported alongside for near-misses; and `vqa_accuracy`, the benchmark's `min(matching human answers / 3, 1)`, which the report does **not** use because with authored accepted answers it degenerates to exact match (one authored answer can never score above 1/3). When accepted answers are supplied the report carries the `exact_match` rate, the mean `anls` and one entry per question, with the verdict `sample-sanity`. On the synthetic path those answers are facts **you drew yourself**, so a high score proves only that the input contract, forward pass and decoding round-trip — and the two recorded misses show what a wrong answer looks like in the report. On BYOD no accepted answers exist, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/blip_vqa_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, golds, sample_kind=sample_kind)
with open('outputs/blip_vqa_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_question')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:12} {metric['value']:.3f}  ({metric['estimation']})")
for entry in report.get('per_question', []):
    print(f"  exact {str(entry['exact_match']):5}  anls {entry['anls']:.2f}  {entry['question']} -> {entry['prediction']!r} (accepted: {entry['golds']})")
if report['verdict'] == 'not-measurable':
    print('No accepted answers exist for these questions, so nothing is scored; read the answers against the image yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every result (question, answer, `new_tokens`, `truncated`, the budget), the evaluation report, the input manifest, the sample identity, digest and accepted answers, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The question/answer pairs are also written as CSV with explicit `image`, `question`, `answer`, `new_tokens`, `truncated` columns, and an annotated PNG shows the image with the questions and answers printed in a panel beneath it for visual inspection (the model returns no location, so nothing is drawn on the image itself) — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

panel_height = 30 + 26 * len(results)
annotated = Image.new('RGB', (image.width, image.height + panel_height), 'white')
annotated.paste(image.convert('RGB'), (0, 0))
draw = ImageDraw.Draw(annotated)
draw.line([(0, image.height + 1), (image.width, image.height + 1)], fill=(120, 120, 120), width=2)
panel_font = ImageFont.load_default(size=16)
for index, result in enumerate(results):
    draw.text((20, image.height + 12 + 26 * index), f"{result['question']}  ->  {result['answer']}", fill=(40, 90, 220), font=panel_font)
annotated.save('outputs/blip_vqa_annotated.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'questions': questions, 'accepted_answers': golds},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/blip_vqa_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/blip_vqa_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'question', 'answer', 'new_tokens', 'truncated'])
    for result in results:
        writer.writerow([image_name, result['question'], result['answer'], result['new_tokens'], result['truncated']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The answers are the text the model generates for an image and a question; nothing in the output scores that text, the model returns no location or evidence, and it answers every question — including one about a blank image — with equal fluency. On the drawn scene the `exact_match` and `anls` values in the evaluation report compare the answers with facts you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, forward pass and decoding work (the repository's smoke run scored 5/7 exact on this drawing, miscounting the trees and miscolouring the door); they say nothing about photographs, cluttered scenes, counting, reading, spatial or commonsense reasoning, or answers longer than a phrase, and a BYOD result is a single-image observation with the verdict `not-measurable`. **The model answers any question about any image** and stops only at end-of-sequence or the token budget: check `truncated`, and treat a plausible answer to an unanswerable question as the expected failure mode, not an exception. The pipeline provides no OCR, no answer localisation, no captioning, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** ask a question the drawing cannot answer (`what is the dog doing?`) and see the model invent one; ask `is it raining?` and `what time of day is it?`; lower `max_new_tokens` to 1 and watch `truncated` turn true on a two-token answer; enable `USE_BYOD` with a photograph you know, type your questions, then pass your own accepted answers to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/blip-vqa-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/blip-vqa-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/blip-vqa-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Salesforce/blip-vqa-base
- Upstream code: https://github.com/salesforce/BLIP
- BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation (Li et al., 2022): https://arxiv.org/abs/2201.12086
- Making the V in VQA Matter — VQA v2 and the VQA accuracy metric (Goyal et al., 2017): https://arxiv.org/abs/1612.00837
- Scene Text Visual Question Answering — the ANLS metric (Biten et al., 2019): https://arxiv.org/abs/1905.13648